# The call graph protocol

A call graph counts calls to component definitions without expanding a physical circuit. Edge weights describe immediate calls. Weighted terminal counts include contributions along every path.

In [ ]:
from IPython.display import SVG, display

from examples.algorithms.call_graph_protocol_catalog import conjunction_graph, protocol_label
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_width import query_resource_logical_width
from flagquantum.drawer.call_graph import (
    export_call_graph_dot, format_call_graph_debug_text,
    format_counts_markdown, render_call_graph_svg,
)

def show_graph(graph, title=protocol_label):
    dot = export_call_graph_dot(
        graph.root,
        graph.direct_calls,
        node_properties=lambda node: {
            "label": title(node), "shape": "box", "style": "rounded,filled",
            "fillcolor": "#eef5f8", "fontname": "Helvetica", "margin": "0.18,0.10",
        },
    )
    display(SVG(data=render_call_graph_svg(dot)))

positive = conjunction_graph((1,) * 6)
def model_label(node):
    qubits = query_resource_logical_width(node)
    ands = query_qec_gate_cost(node).and_bloq
    return f"{protocol_label(node)}\nLogical qubits {qubits} · ANDs {ands}"

show_graph(positive, title=model_label)

## Query a native graph

A graph exposes the root, direct call weights, and aggregated terminal counts.

In [ ]:
print(format_call_graph_debug_text(positive.root, positive.direct_calls, title=protocol_label))
print(format_counts_markdown(positive.leaf_counts, title=protocol_label))

Query the declaration directly when only the immediate callees are needed. A binary AND is a terminal component in this model.

In [ ]:
from examples.algorithms.call_graph_protocol_catalog import BINARY_AND, resource_declaration
from flagquantum.compiler._component_trace.call_graph import component_callee_counts

assert component_callee_counts(BINARY_AND, resource_declaration) == {}
component_callee_counts(BINARY_AND, resource_declaration)

A generalizer can combine definitions that differ in attributes outside the selected cost model. Compare the mixed-polarity conjunction to the all-positive one.

In [ ]:
mixed = conjunction_graph((1, 0) * 3)
show_graph(mixed)
print(format_counts_markdown(mixed.leaf_counts, title=protocol_label))

Generalize the polarity of the binary AND with two allocated symbols. The mixed-polarity parent's explicit X count remains visible.

In [ ]:
generalized = conjunction_graph((1, 0) * 3, generalize_activation=True)
show_graph(generalized)
print(format_counts_markdown(generalized.leaf_counts, title=protocol_label))

The original notebook describes a reduction in the explicit Clifford count. In the pinned example the six X calls remain declared after the binary AND rewrite, so the two counts above retain them.

## Additional graph options

Use `keep` to stop at selected components and `max_depth` to bound traversal. The first visit determines the expansion of a shared node.

## Define symbolic callees

A declaration callback receives the query's symbol allocator. This example declares two AND calls per element and five CNOTs, without providing a physical implementation.

In [ ]:
from sympy import Symbol

from examples.algorithms.call_graph_protocol_catalog import declared_function_graph
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.symbolic_math import asymptotic_order

n = Symbol("n")
declared = declared_function_graph(n)
limited = symbolic_call_graph(positive.root, resource_declaration, max_depth=0)
assert limited.leaf_counts == {positive.root: 1}

In [ ]:
show_graph(declared)
assert declared.estimate_leaf_cost(lambda node: int(node == BINARY_AND)) == 2 * n
print("Declared AND asymptotic order", asymptotic_order(2 * n))

## Count a stored decomposition

In [ ]:
from examples.algorithms.call_graph_protocol_catalog import swap_decomposition_program
from flagquantum.compiler._component_trace.program_call_graph import program_component_call_graph

program = swap_decomposition_program()

The typed Program graph derives the three calls from the stored SWAP decomposition. Append the bound gate stream to inspect its physical instruction count.

In [ ]:
from collections import Counter

from examples.algorithms.binary_field_arithmetic import FieldOperation
from flagquantum.circuit import Circuit
from flagquantum.compiler.registers.register_binding import bind_register_program

swap_graph = program_component_call_graph(program)
show_graph(swap_graph, title=lambda name: "SWAP" if name == "main" else name)
assert swap_graph.leaf_counts == {"CNOT": 3}
bound = bind_register_program(program, ((0,), (1,)))
circuit = Circuit(2).append(FieldOperation(2, bound.circuit.instructions))
print("Primitive counts", dict(Counter(gate.name for gate in circuit.to_ir().instructions)))

## Check model and decomposition agreement

Compare independently declared AND counts and ladder stages after applying the same policy to both.

In [ ]:
from examples.algorithms.call_graph_protocol_catalog import conjunction_counts_agree

assert conjunction_counts_agree((1, 0) * 3)
print("Declared and decomposed conjunction counts agree")